In [1]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import datetime

sys.path.insert(0, "cellular_cycle")

from cell_cycle_death_model import model
from model_fxns import *
from pysb.simulator import ScipyOdeSimulator
from pysb.simulator import BngSimulator


# Where to save large simulation output files
SAVE_DIR = Path(Path.home(),'ECM_mod_output')
SAVE_DIR.mkdir(exist_ok=True)

OVERWRITE = True # Overwrite existing simulation files

CURDATE = datetime.datetime.now().strftime('%Y%m%d')

SET_JPARAM = False # Set the parameter values for the simulation using PSO-optimized values

### Define timesteps

In [2]:
t = np.linspace(0, 20000, 200001)

### Load model and update parameters

In [3]:
# Importing the parameters:
new_param = pd.read_csv('model_data/parameters.csv')
for key in new_param.keys():
    try:
        if model.parameters[key].value != new_param[key][0]:
            print(f'Changing {key} from {model.parameters[key].value} to {new_param[key][0]}')
            model.parameters[key].value = new_param[key]
    except:
        print(f'{key} not found in model parameters')

In [4]:
if SET_JPARAM:
    filenames = [Path(SAVE_DIR,f) for f in os.listdir(SAVE_DIR) if 'stoch_sim' in f and '_psoparam_cycdi_0_0' in f]
else:
    filenames = [Path(SAVE_DIR,f) for f in os.listdir(SAVE_DIR) if 'stoch_sim' in f and '_param_cycdi_0_0' in f]
filenames = sorted(filenames)

file_order = (2,3,0,1,4,5)

stoch_filenames = [filenames[i] for i in file_order]
stoch_filenames

[PosixPath('/Users/darren/ECM_mod_output/stoch_sim_soft_DMSO_param_cycdi_0_0.csv'),
 PosixPath('/Users/darren/ECM_mod_output/stoch_sim_soft_erl_param_cycdi_0_0.csv'),
 PosixPath('/Users/darren/ECM_mod_output/stoch_sim_int_DMSO_param_cycdi_0_0.csv'),
 PosixPath('/Users/darren/ECM_mod_output/stoch_sim_int_erl_param_cycdi_0_0.csv'),
 PosixPath('/Users/darren/ECM_mod_output/stoch_sim_stiff_DMSO_param_cycdi_0_0.csv'),
 PosixPath('/Users/darren/ECM_mod_output/stoch_sim_stiff_erl_param_cycdi_0_0.csv')]

### Define stiffness values

In [5]:
vals = [0, 29, 49, 74, 99]

### Hardcoding AKT, ERK and FAK input values for erlotinib
Scaling values between low and high stiffness in 100 steps; corresponding stiffness values = 0-99.

In [6]:
treatment = 'ERL'
FAKs = list(np.linspace(0.4,0.6,50))
FAKs.extend(list(np.linspace(0.6,1,50)))
AKTs = list(np.linspace(0.3,0.6,50))
AKTs.extend([0.6]*50)
ERKs = [0.2]*100

### Run simulations and save output

In [7]:
for val in vals:
    model.parameters['FAK'].value = FAKs[val]
    model.parameters['AKT'].value = AKTs[val]
    model.parameters['ERK'].value = ERKs[val]
    
    num = f'{val:03}'
    
    simres = ScipyOdeSimulator(model, t, verbose=False, compiler='cython').run()#deterministic simulation
    yout = simres.observables
    ode_filename = Path(SAVE_DIR, f'ODE_sim_{treatment}_stiff_{num}_orig.csv')
    if ode_filename.exists():
        print(f"ODE simulation already exists at {ode_filename}")
    if not ode_filename.exists() or OVERWRITE:
        np.savetxt(ode_filename, yout, delimiter=',')
        print(f"ODE simulation saved to {ode_filename}")
    
    sim = BngSimulator(model, tspan=t, verbose=False) #stochastic simulator
    simres = sim.run(method='ssa', verbose=False, seed=33) #running the simulation; remove seed for truly random; seed=33 is used for similarity to deterministic sim
    yout = simres.observables
    stoch_filename = Path(SAVE_DIR, f'stoch_sim_{treatment}_stiff_{num}_orig.csv')
    if stoch_filename.exists():
        print(f"Stochastic simulation already exists at {stoch_filename}")
    if not stoch_filename.exists() or OVERWRITE:
        np.savetxt(stoch_filename, yout, delimiter=',')
        print(f"Stochastic simulation saved to {stoch_filename}")

ld: warning: duplicate -rpath '/Users/darren/mambaforge/envs/ECM_model/lib' ignored


ODE simulation already exists at /Users/darren/ECM_mod_output/ODE_sim_ERL_stiff_000_orig.csv
ODE simulation saved to /Users/darren/ECM_mod_output/ODE_sim_ERL_stiff_000_orig.csv
Stochastic simulation already exists at /Users/darren/ECM_mod_output/stoch_sim_ERL_stiff_000_orig.csv
Stochastic simulation saved to /Users/darren/ECM_mod_output/stoch_sim_ERL_stiff_000_orig.csv
ODE simulation already exists at /Users/darren/ECM_mod_output/ODE_sim_ERL_stiff_029_orig.csv
ODE simulation saved to /Users/darren/ECM_mod_output/ODE_sim_ERL_stiff_029_orig.csv
Stochastic simulation already exists at /Users/darren/ECM_mod_output/stoch_sim_ERL_stiff_029_orig.csv
Stochastic simulation saved to /Users/darren/ECM_mod_output/stoch_sim_ERL_stiff_029_orig.csv
ODE simulation already exists at /Users/darren/ECM_mod_output/ODE_sim_ERL_stiff_049_orig.csv
ODE simulation saved to /Users/darren/ECM_mod_output/ODE_sim_ERL_stiff_049_orig.csv
Stochastic simulation already exists at /Users/darren/ECM_mod_output/stoch_sim_

### Hardcoding AKT, ERK and FAK input values for DMSO
Scaling values between low and high stiffness in 100 steps; corresponding stiffness values = 0-99.

In [8]:
treatment = 'DMSO'
FAKs = list(np.linspace(0.4,0.6,50))
FAKs.extend(list(np.linspace(0.6,1,50)))
AKTs = list(np.linspace(0.6,1,50))
AKTs.extend([1]*50)
ERKs = [1]*100

### Run simulations and save output

In [9]:
for val in vals:
    model.parameters['FAK'].value = FAKs[val]
    model.parameters['AKT'].value = AKTs[val]
    model.parameters['ERK'].value = ERKs[val]
    
    num = f'{val:03}'
    
    simres = ScipyOdeSimulator(model, t, verbose=False, compiler='cython').run()#deterministic simulation
    yout = simres.observables
    ode_filename = Path(SAVE_DIR, f'ODE_sim_{treatment}_stiff_{num}_orig.csv')
    if ode_filename.exists():
        print(f"ODE simulation already exists at {ode_filename}")
    if not ode_filename.exists() or OVERWRITE:
        np.savetxt(ode_filename, yout, delimiter=',')
        print(f"ODE simulation saved to {ode_filename}")
    
    sim = BngSimulator(model, tspan=t, verbose=False) #stochastic simulator
    simres = sim.run(method='ssa', verbose=False, seed=33) #running the simulation; remove seed for truly random; seed=33 is used for similarity to deterministic sim
    yout = simres.observables
    stoch_filename = Path(SAVE_DIR, f'stoch_sim_{treatment}_stiff_{num}_orig.csv')
    if stoch_filename.exists():
        print(f"Stochastic simulation already exists at {stoch_filename}")
    if not stoch_filename.exists() or OVERWRITE:
        np.savetxt(stoch_filename, yout, delimiter=',')
        print(f"Stochastic simulation saved to {stoch_filename}")



ODE simulation already exists at /Users/darren/ECM_mod_output/ODE_sim_DMSO_stiff_000_orig.csv
ODE simulation saved to /Users/darren/ECM_mod_output/ODE_sim_DMSO_stiff_000_orig.csv
Stochastic simulation already exists at /Users/darren/ECM_mod_output/stoch_sim_DMSO_stiff_000_orig.csv
Stochastic simulation saved to /Users/darren/ECM_mod_output/stoch_sim_DMSO_stiff_000_orig.csv
ODE simulation already exists at /Users/darren/ECM_mod_output/ODE_sim_DMSO_stiff_029_orig.csv
ODE simulation saved to /Users/darren/ECM_mod_output/ODE_sim_DMSO_stiff_029_orig.csv
Stochastic simulation already exists at /Users/darren/ECM_mod_output/stoch_sim_DMSO_stiff_029_orig.csv
Stochastic simulation saved to /Users/darren/ECM_mod_output/stoch_sim_DMSO_stiff_029_orig.csv
ODE simulation already exists at /Users/darren/ECM_mod_output/ODE_sim_DMSO_stiff_049_orig.csv
ODE simulation saved to /Users/darren/ECM_mod_output/ODE_sim_DMSO_stiff_049_orig.csv
Stochastic simulation already exists at /Users/darren/ECM_mod_output/